In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

In [ ]:

DATASET_DIR = "/kaggle/input/datasets/iamsouravbanerjee/animal-image-dataset-90-different-animals/animals"  # Path where the 90 animal folders are extracted
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
EPOCHS = 15


CARNIVORES = {
    "bear", "cheetah", "coyote", "crocodile", "fox", "hyena", 
    "jaguar", "leopard", "lion", "panther", "tiger", "wolf"
}

In [ ]:

def load_dataset_dataframe(base_dir):
    file_paths = []
    labels = []
    
    # Traverse directory 
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                filepath = os.path.join(root, file)
                label = os.path.basename(root).lower().strip()
                file_paths.append(filepath)
                labels.append(label)
                
    return pd.DataFrame({'filepath': file_paths, 'label': labels})

df = load_dataset_dataframe(DATASET_DIR)
print(f"Total images found: {len(df)}")
print(f"Total categories: {df['label'].nunique()}")

In [ ]:
# Visualize Initial Class Distribution
plt.figure(figsize=(12, 5))
top_classes = df['label'].value_counts().head(20)
sns.barplot(x=top_classes.index, y=top_classes.values, palette="Blues_r")
plt.xticks(rotation=45, ha="right")
plt.title("Initial Class Distribution (Imbalanced Sample)")
plt.xlabel("Animal Class")
plt.ylabel("Image Count")
plt.tight_layout()
plt.savefig("initial_distribution.png")
plt.show()
plt.close()

In [ ]:

#  BALANCE DATASET 
max_samples = df['label'].value_counts().max()
df_balanced_list = []

for label, group in df.groupby('label'):
    oversampled_group = resample(group, replace=True, n_samples=max_samples, random_state=42)
    df_balanced_list.append(oversampled_group)

df_balanced = pd.concat(df_balanced_list).sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Visualize Balanced Class Distribution
plt.figure(figsize=(12, 5))
balanced_counts = df_balanced['label'].value_counts().head(20)
sns.barplot(x=balanced_counts.index, y=balanced_counts.values, palette="Greens_r")
plt.xticks(rotation=45, ha="right")
plt.title("Balanced Class Distribution After Oversampling")
plt.xlabel("Animal Class")
plt.ylabel("Image Count")
plt.tight_layout()
plt.savefig("balanced_distribution.png")
plt.show()
plt.close()

In [ ]:
# Encode class names 
class_names = sorted(df_balanced['label'].unique())
label_to_idx = {name: idx for idx, name in enumerate(class_names)}
df_balanced['label_idx'] = df_balanced['label'].map(label_to_idx)


np.save("classes.npy", class_names)

In [ ]:
# PREPROCESSING & AUGMENTATION 
train_df, val_df = train_test_split(
    df_balanced, test_size=0.2, random_state=42, stratify=df_balanced['label_idx']
)

def process_path(filepath, label):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = img / 255.0 
    return img, label

def build_tf_dataset(df_data, is_training=True):
    ds = tf.data.Dataset.from_tensor_slices((df_data['filepath'].values, df_data['label_idx'].values))
    ds = ds.map(process_path, num_parallel_calls=tf.data.AUTOTUNE)
    if is_training:
        ds = ds.shuffle(buffer_size=1000)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = build_tf_dataset(train_df, is_training=True)
val_ds = build_tf_dataset(val_df, is_training=False)


data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1)
])

In [ ]:
#  BUILD CNN MODEL 
inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = data_augmentation(inputs)

x = layers.Conv2D(32, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(64, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(128, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Conv2D(256, (3, 3), activation='relu')(x)
x = layers.MaxPooling2D((2, 2))(x)

x = layers.Flatten()(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(len(class_names), activation='softmax')(x)

model = models.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
#  TRAIN & EVALUATE MODEL 
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS
)

model.save("animal_cnn_model.h5")

In [ ]:
model.save("animal_cnn_model.keras")

In [ ]:
#  ACCURACY & LOSS GRAPH
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['accuracy'], label='Train Accuracy', color='blue')
ax1.plot(history.history['val_accuracy'], label='Val Accuracy', color='orange')
ax1.set_title('Model Accuracy')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'], label='Train Loss', color='blue')
ax2.plot(history.history['val_loss'], label='Val Loss', color='orange')
ax2.set_title('Model Loss')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()

plt.tight_layout()
plt.savefig("accuracy_loss_graph.png")
plt.show()